# Module 2.2 — Cleaning and Validating Messy Data

## Preparing a fictional appointments dataset for analysis

Real datasets rarely arrive ready for exploratory data analysis. In this lesson, we will clean a deliberately messy clinic-administration dataset while preserving evidence of every decision.

> **Privacy note:** Every record in this notebook is fictional. The dataset contains no real patient information and must not be used for clinical decisions.

## Learning objectives

By the end of this notebook, you will be able to:

- Preserve raw data and create a separate working copy.
- Detect missing values, duplicates, inconsistent categories, and invalid values.
- Standardise column names and text with pandas string methods.
- Convert text to numeric, date, and Boolean data types safely.
- Compare `dropna()` and `fillna()` without applying them automatically.
- Remove confirmed exact duplicates while retaining an audit trail.
- Validate ranges, categories, identifiers, and relationships between fields.
- Create row-level quality flags and a data-quality log.
- Export cleaned data and verify the saved result.
- Explain why cleaning decisions affect EDA and machine learning.

## Lesson roadmap

| Part | Focus |
|---|---|
| 1 | Load and profile messy data |
| 2 | Preserve the raw dataset |
| 3 | Standardise columns and text |
| 4 | Convert missing values and data types |
| 5 | Investigate duplicates |
| 6 | Decide how to handle missing data |
| 7 | Validate business rules |
| 8 | Record decisions and export clean data |
| 9 | Practice and knowledge check |

**Working pattern:** inspect → state the rule → transform a copy → validate the result → record the decision.

## 1. Why cleaning matters

> A calculation can run successfully and still be wrong. If one department appears as `Physiotherapy`, `physio`, and ` PHYSIO `, pandas will treat those labels as different groups. If a duration contains the word `forty`, a numerical calculation may fail or silently exclude the record after conversion.

Cleaning is not about making data look tidy. It is about making the representation consistent with a documented meaning while keeping uncertainty visible.

### Connections to EDA and machine learning

- In **EDA**, inconsistent labels split counts and summaries into false categories.
- In **feature engineering**, text-formatted numbers prevent reliable arithmetic.
- In **machine learning**, inconsistent categories can become separate model inputs.
- In **evaluation**, duplicate records can make performance appear better than it is.
- In every stage, undocumented row removal can bias results.

## 2. Setup

We need pandas and the standard-library `Path` class. If pandas is unavailable in this notebook's environment, install it with:

```text
python -m pip install pandas
```

In [ ]:
from pathlib import Path
import pandas as pd

print("pandas version:", pd.__version__)

## 3. Create the fictional raw file

The following records intentionally include:

- Leading and trailing spaces.
- Inconsistent capitalisation and abbreviations.
- Placeholder strings such as `not recorded` and `unknown`.
- An invalid date.
- A text value in a numerical field.
- A negative waiting time.
- An appointment date before its booking date.
- An exact duplicate row.

This self-contained setup ensures that everyone works with the same teaching data.

In [ ]:
raw_records = [
    {
        "Appointment ID": " apt-1001 ",
        "Booking Date": "2026-09-01",
        "Appointment Date": "2026-09-05",
        "Department": " general practice ",
        "Appointment Type": "initial",
        "Wait Minutes": "12",
        "Scheduled Minutes": "20",
        "Status": "completed",
        "Reminder Sent": "yes",
        "Booking Channel": "phone",
    },
    {
        "Appointment ID": "APT-1002",
        "Booking Date": "2026-09-01",
        "Appointment Date": "2026-09-06",
        "Department": "PHYSIO",
        "Appointment Type": "follow up",
        "Wait Minutes": "25",
        "Scheduled Minutes": "30",
        "Status": "Complete",
        "Reminder Sent": "YES",
        "Booking Channel": "online",
    },
    {
        "Appointment ID": "APT-1003",
        "Booking Date": "2026-09-02",
        "Appointment Date": "2026-09-06",
        "Department": "derm ",
        "Appointment Type": "Initial",
        "Wait Minutes": "8",
        "Scheduled Minutes": "20",
        "Status": "COMPLETED",
        "Reminder Sent": "no",
        "Booking Channel": "Phone",
    },
    {
        "Appointment ID": "APT-1004",
        "Booking Date": "2026-09-02",
        "Appointment Date": "2026-09-07",
        "Department": "General Practice",
        "Appointment Type": "follow-up",
        "Wait Minutes": "not recorded",
        "Scheduled Minutes": "15",
        "Status": "cancelled",
        "Reminder Sent": "yes",
        "Booking Channel": "ONLINE",
    },
    {
        "Appointment ID": "APT-1005",
        "Booking Date": "2026-09-03",
        "Appointment Date": "2026-09-08",
        "Department": "Physiotherapy",
        "Appointment Type": "Initial",
        "Wait Minutes": "40",
        "Scheduled Minutes": "forty",
        "Status": "Completed",
        "Reminder Sent": "Y",
        "Booking Channel": "front desk",
    },
    {
        "Appointment ID": "APT-1006",
        "Booking Date": "2026-09-03",
        "Appointment Date": "2026-09-08",
        "Department": "GENERAL PRACTICE",
        "Appointment Type": "Initial",
        "Wait Minutes": "15",
        "Scheduled Minutes": "20",
        "Status": "no show",
        "Reminder Sent": "N",
        "Booking Channel": "phone",
    },
    {
        "Appointment ID": "APT-1007",
        "Booking Date": "2026-09-04",
        "Appointment Date": "2026-09-09",
        "Department": "Dermatology",
        "Appointment Type": "Follow Up",
        "Wait Minutes": "18",
        "Scheduled Minutes": "15",
        "Status": "completed ",
        "Reminder Sent": "true",
        "Booking Channel": "Online",
    },
    {
        "Appointment ID": "APT-1008",
        "Booking Date": "2026-09-04",
        "Appointment Date": "2026-09-10",
        "Department": "general practice",
        "Appointment Type": "initial",
        "Wait Minutes": "6",
        "Scheduled Minutes": "20",
        "Status": "completed",
        "Reminder Sent": "yes",
        "Booking Channel": "reception",
    },
    {
        "Appointment ID": "APT-1008",
        "Booking Date": "2026-09-04",
        "Appointment Date": "2026-09-10",
        "Department": "general practice",
        "Appointment Type": "initial",
        "Wait Minutes": "6",
        "Scheduled Minutes": "20",
        "Status": "completed",
        "Reminder Sent": "yes",
        "Booking Channel": "reception",
    },
    {
        "Appointment ID": "APT-1009",
        "Booking Date": "2026-09-05",
        "Appointment Date": "2026-09-10",
        "Department": " physio ",
        "Appointment Type": "Follow-up",
        "Wait Minutes": "-4",
        "Scheduled Minutes": "30",
        "Status": "completed",
        "Reminder Sent": "1",
        "Booking Channel": "web",
    },
    {
        "Appointment ID": "APT-1010",
        "Booking Date": "2026-09-05",
        "Appointment Date": "2026-09-31",
        "Department": "Dermatology",
        "Appointment Type": "Initial",
        "Wait Minutes": "N/A",
        "Scheduled Minutes": "20",
        "Status": "canceled",
        "Reminder Sent": "0",
        "Booking Channel": "phone",
    },
    {
        "Appointment ID": "APT-1011",
        "Booking Date": "2026-09-12",
        "Appointment Date": "2026-09-11",
        "Department": "General Practice",
        "Appointment Type": "Follow-up",
        "Wait Minutes": "10",
        "Scheduled Minutes": "15",
        "Status": "completed",
        "Reminder Sent": "yes",
        "Booking Channel": "unknown",
    },
    {
        "Appointment ID": "APT-1012",
        "Booking Date": "2026-09-06",
        "Appointment Date": "2026-09-12",
        "Department": "Physiotherapy",
        "Appointment Type": "Initial",
        "Wait Minutes": "30",
        "Scheduled Minutes": "45",
        "Status": "Missed",
        "Reminder Sent": "yes",
        "Booking Channel": "Reception",
    },
    {
        "Appointment ID": "APT-1013",
        "Booking Date": "",
        "Appointment Date": "2026-09-13",
        "Department": "DERM",
        "Appointment Type": "initial",
        "Wait Minutes": "22",
        "Scheduled Minutes": "20",
        "Status": "completed",
        "Reminder Sent": "not recorded",
        "Booking Channel": "online",
    },
]

raw_source_df = pd.DataFrame(raw_records)
raw_path = Path("fictional_appointments_messy.csv")
raw_source_df.to_csv(raw_path, index=False)

print(f"Created {raw_path} with {len(raw_source_df)} rows.")

## 4. Load without silently changing placeholders

For this exercise, `keep_default_na=False` keeps strings such as `N/A` and empty text visible. We will decide explicitly which strings should become missing values.

> Import settings are part of the cleaning process. If software interprets placeholders automatically, record that behaviour in your workflow.

In [ ]:
raw_df = pd.read_csv(raw_path, keep_default_na=False)
raw_df.head()

## 5. Profile before changing anything

Before cleaning, inspect the raw table. This gives us a baseline against which we can check every transformation.

In [ ]:
print("Shape:", raw_df.shape)
print("Columns:", raw_df.columns.tolist())
print("\nData types:")
print(raw_df.dtypes)
print("\nBlank strings by column:")
print(raw_df.eq("").sum())

In [ ]:
# Frequency tables expose spelling, case, and whitespace differences.
for column in ["Department", "Appointment Type", "Status", "Reminder Sent", "Booking Channel"]:
    print(f"\n--- {column} ---")
    print(raw_df[column].value_counts(dropna=False))

### Pause and notice

Write down at least five problems before viewing later transformations.

- **Problem 1:** 
- **Problem 2:** 
- **Problem 3:** 
- **Problem 4:** 
- **Problem 5:** 

For each issue, distinguish between:

- A **representation problem**, such as inconsistent capitalisation.
- A **missingness problem**, such as an absent booking date.
- A **validity problem**, such as a negative waiting time.
- A **uniqueness problem**, such as a repeated appointment ID.

## 6. Preserve the raw data

Never make the only available copy of raw data your working table. We create a deep copy and leave `raw_df` unchanged.

In [ ]:
clean_df = raw_df.copy(deep=True)

print("Raw object ID:", id(raw_df))
print("Clean object ID:", id(clean_df))
print("Same object?", raw_df is clean_df)

### Start a data-quality log

A log records what changed, how many records were affected, and why the decision was made. This makes the work reviewable and reproducible.

In [ ]:
quality_log = []


def record_quality_step(step, affected_rows, decision):
    """Append one cleaning decision to the data-quality log."""
    quality_log.append({
        "step": step,
        "affected_rows": int(affected_rows),
        "decision": decision,
    })


record_quality_step(
    "Load raw data",
    len(clean_df),
    "Preserved the source table and created a separate working copy.",
)

## 7. Standardise column names

Consistent `snake_case` names are easier to type and less error-prone. The operations below:

1. Remove leading and trailing whitespace.
2. Convert text to lowercase.
3. Replace spaces with underscores.

In [ ]:
old_columns = clean_df.columns.tolist()

clean_df.columns = (
    clean_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

new_columns = clean_df.columns.tolist()
print("Before:", old_columns)
print("After: ", new_columns)

record_quality_step(
    "Standardise column names",
    len(clean_df.columns),
    "Applied lowercase snake_case labels to all columns.",
)

## 8. Replace placeholder strings

Different systems use different missing-value markers. Here we define the markers that mean “no recorded value” in this fictional file.

> Replacing `unknown` with a missing value is a documented choice for this dataset. In another dataset, `Unknown` might be a valid reporting category rather than missing data.

In [ ]:
missing_markers = ["", "N/A", "not recorded", "unknown"]

placeholder_count = clean_df.isin(missing_markers).sum().sum()
clean_df = clean_df.replace(missing_markers, pd.NA)

print("Placeholder cells converted to missing:", placeholder_count)
print("\nMissing values by column:")
print(clean_df.isna().sum())

record_quality_step(
    "Convert placeholders to missing values",
    placeholder_count,
    "Mapped documented placeholder strings to pd.NA.",
)

## 9. Clean identifiers

Appointment IDs should follow one consistent format. We strip whitespace and convert letters to uppercase.

In [ ]:
original_ids = clean_df["appointment_id"].copy()

clean_df["appointment_id"] = (
    clean_df["appointment_id"]
    .astype("string")
    .str.strip()
    .str.upper()
)

changed_ids = original_ids.fillna("<NA>").ne(
    clean_df["appointment_id"].fillna("<NA>")
).sum()

print("Changed IDs:", changed_ids)
print(clean_df["appointment_id"].head())

record_quality_step(
    "Standardise appointment IDs",
    changed_ids,
    "Removed surrounding spaces and converted letters to uppercase.",
)

### Validate the ID pattern

The fictional rule is `APT-` followed by exactly four digits. `str.fullmatch()` checks the whole value against that pattern.

In [ ]:
valid_id_pattern = clean_df["appointment_id"].str.fullmatch(r"APT-\d{4}", na=False)

print("Invalid ID count:", (~valid_id_pattern).sum())
clean_df.loc[~valid_id_pattern, ["appointment_id"]]

## 10. Standardise categories

A category should have one canonical label. We first strip whitespace and convert to lowercase. We then map known aliases to preferred labels.

Do not map unfamiliar values by guesswork. Keep them visible for investigation.

In [ ]:
category_columns = [
    "department",
    "appointment_type",
    "status",
    "booking_channel",
]

for column in category_columns:
    clean_df[column] = clean_df[column].astype("string").str.strip().str.lower()

print(clean_df[category_columns].head())

In [ ]:
department_map = {
    "general practice": "General Practice",
    "physio": "Physiotherapy",
    "physiotherapy": "Physiotherapy",
    "derm": "Dermatology",
    "dermatology": "Dermatology",
}

appointment_type_map = {
    "initial": "Initial",
    "follow up": "Follow-up",
    "follow-up": "Follow-up",
}

status_map = {
    "complete": "Completed",
    "completed": "Completed",
    "cancelled": "Cancelled",
    "canceled": "Cancelled",
    "missed": "Missed",
    "no show": "Missed",
}

channel_map = {
    "phone": "Phone",
    "online": "Online",
    "web": "Online",
    "front desk": "Reception",
    "reception": "Reception",
}

mappings = {
    "department": department_map,
    "appointment_type": appointment_type_map,
    "status": status_map,
    "booking_channel": channel_map,
}

for column, mapping in mappings.items():
    before = clean_df[column].copy()
    clean_df[column] = clean_df[column].map(mapping)
    changed = before.fillna("<NA>").ne(clean_df[column].fillna("<NA>")).sum()
    record_quality_step(
        f"Standardise {column}",
        changed,
        f"Mapped known aliases to {sorted(set(mapping.values()))}.",
    )

clean_df[category_columns].head()

### Check categories after mapping

A mapping can accidentally create missing values when an unexpected label is not present in the mapping dictionary. Always inspect the result.

In [ ]:
for column in category_columns:
    print(f"\n--- {column} ---")
    print(clean_df[column].value_counts(dropna=False))

## 11. Convert a yes/no field

The raw `reminder_sent` column uses several representations. We map recognised values to `True` or `False`, then use pandas' nullable Boolean type so an unrecorded value can remain missing.

In [ ]:
reminder_map = {
    "yes": True,
    "y": True,
    "true": True,
    "1": True,
    "no": False,
    "n": False,
    "false": False,
    "0": False,
}

reminder_text = clean_df["reminder_sent"].astype("string").str.strip().str.lower()
clean_df["reminder_sent"] = reminder_text.map(reminder_map).astype("boolean")

print(clean_df["reminder_sent"].value_counts(dropna=False))

record_quality_step(
    "Convert reminder_sent",
    clean_df["reminder_sent"].notna().sum(),
    "Mapped recognised yes/no representations to nullable Boolean values.",
)

## 12. Convert numerical fields safely

`pd.to_numeric(..., errors="coerce")` converts valid numeric text and turns invalid text into a missing value. Coercion is useful only when followed by an explicit check of what failed.

Before converting, save the original text. That allows us to identify newly missing values created by conversion.

In [ ]:
numeric_columns = ["wait_minutes", "scheduled_minutes"]

for column in numeric_columns:
    original = clean_df[column].copy()
    converted = pd.to_numeric(original, errors="coerce")

    conversion_failures = original.notna() & converted.isna()
    print(f"\n{column} conversion failures:")
    display(clean_df.loc[conversion_failures, ["appointment_id", column]])

    clean_df[column] = converted.astype("Int64")

    record_quality_step(
        f"Convert {column} to numeric",
        conversion_failures.sum(),
        "Converted valid text to nullable integers; invalid text became missing and was retained for review.",
    )

clean_df[numeric_columns].dtypes

> Coercion does not prove that a number is reasonable. The value `-4` is numeric, but a negative waiting time violates our fictional business rule. Type conversion and validity checking are separate steps.

## 13. Convert date fields safely

The expected format is `YYYY-MM-DD`. Invalid values become `NaT`, pandas' missing value for date/time data.

In [ ]:
date_columns = ["booking_date", "appointment_date"]

for column in date_columns:
    original = clean_df[column].copy()
    converted = pd.to_datetime(
        original,
        format="%Y-%m-%d",
        errors="coerce",
    )

    conversion_failures = original.notna() & converted.isna()
    print(f"\n{column} conversion failures:")
    display(clean_df.loc[conversion_failures, ["appointment_id", column]])

    clean_df[column] = converted

    record_quality_step(
        f"Convert {column} to datetime",
        conversion_failures.sum(),
        "Parsed YYYY-MM-DD values; invalid dates became NaT and were retained for review.",
    )

clean_df[date_columns].dtypes

### Derive a booking lead-time field

Once dates are real datetime values, we can subtract them. The result below is the number of days between booking and appointment.

In [ ]:
clean_df["lead_days"] = (
    clean_df["appointment_date"] - clean_df["booking_date"]
).dt.days.astype("Int64")

clean_df[["appointment_id", "booking_date", "appointment_date", "lead_days"]].head()

## 14. Investigate duplicates

There are at least two different questions:

1. Are any **entire rows** exact copies?
2. Does the intended **identifier** occur more than once?

Never remove duplicates merely because two rows look similar. Two genuine appointments may share a department, date, duration, and status.

In [ ]:
exact_duplicate_mask = clean_df.duplicated(keep=False)
id_duplicate_mask = clean_df.duplicated(subset="appointment_id", keep=False)

print("Rows belonging to exact duplicate groups:", exact_duplicate_mask.sum())
print("Rows with a repeated appointment ID:", id_duplicate_mask.sum())

display(clean_df.loc[id_duplicate_mask].sort_values("appointment_id"))

### Remove only the confirmed exact duplicate

`drop_duplicates()` keeps the first copy by default. We record the row count before and after the operation.

In [ ]:
rows_before_deduplication = len(clean_df)
clean_df = clean_df.drop_duplicates().reset_index(drop=True)
rows_removed = rows_before_deduplication - len(clean_df)

print("Rows removed:", rows_removed)
print("Remaining rows:", len(clean_df))
print("Duplicate appointment IDs remaining:", clean_df["appointment_id"].duplicated().sum())

record_quality_step(
    "Remove confirmed exact duplicates",
    rows_removed,
    "Kept the first copy of each fully identical row.",
)

## 15. Understand missing data before acting

Count and calculate the percentage of missing values. A missing-value percentage provides scale, but it does not tell us *why* values are missing.

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": clean_df.isna().sum(),
    "missing_percent": clean_df.isna().mean().mul(100).round(1),
}).sort_values("missing_count", ascending=False)

missing_summary

### Inspect the context of missingness

A missing waiting time may mean something different for a cancelled appointment than for a completed appointment. View relevant columns together before choosing an action.

In [ ]:
rows_with_missing_values = clean_df.loc[
    clean_df.isna().any(axis=1),
    [
        "appointment_id",
        "booking_date",
        "appointment_date",
        "status",
        "wait_minutes",
        "scheduled_minutes",
        "reminder_sent",
        "booking_channel",
    ],
]

rows_with_missing_values

## 16. Compare missing-data options

There is no universal command that “fixes” missing values.

| Option | Possible use | Main risk |
|---|---|---|
| Keep missing | Preserve uncertainty for investigation | Some later methods require handling |
| Drop rows | Remove unusable observations | Lose information and possibly introduce bias |
| Drop columns | Remove an unusable field | Lose a potentially useful feature |
| Fill with a constant | Create an explicit category such as `Unknown` | May be mistaken for an observed value |
| Fill with a statistic | Support some numerical workflows | Can distort distributions and relationships |

The following cells create demonstrations. They do **not** overwrite `clean_df`.

In [ ]:
# Demonstration only: drop rows missing a booking date.
demo_dropped = clean_df.dropna(subset=["booking_date"])

print("Original rows:", len(clean_df))
print("Rows after dropna:", len(demo_dropped))

In [ ]:
# Demonstration only: make missing booking channels explicit for reporting.
demo_filled_channel = clean_df.copy()
demo_filled_channel["booking_channel"] = (
    demo_filled_channel["booking_channel"].fillna("Unknown")
)

demo_filled_channel["booking_channel"].value_counts(dropna=False)

In [ ]:
# Demonstration only: median imputation changes the observed wait-time distribution.
demo_filled_wait = clean_df.copy()
median_wait = demo_filled_wait["wait_minutes"].median()
demo_filled_wait["wait_minutes"] = demo_filled_wait["wait_minutes"].fillna(median_wait)

print("Median used:", median_wait)
print("Missing before:", clean_df["wait_minutes"].isna().sum())
print("Missing after:", demo_filled_wait["wait_minutes"].isna().sum())

### Decision for this lesson

We will:

- Keep missing dates, durations, and waiting times visible.
- Fill missing `booking_channel` with `Unknown` because it is used as an explicit reporting category here.
- Keep the missing reminder value as nullable Boolean data.
- Add validity flags rather than silently dropping questionable rows.

This is a teaching decision for the fictional scenario, not a rule for all datasets.

In [ ]:
missing_channels_before = clean_df["booking_channel"].isna().sum()
clean_df["booking_channel"] = clean_df["booking_channel"].fillna("Unknown")

record_quality_step(
    "Label missing booking channels",
    missing_channels_before,
    "Filled missing channel values with the explicit reporting category 'Unknown'.",
)

## 17. Validate allowed categories

A business rule states which labels are acceptable. Values outside the allowed set should be flagged, not guessed.

In [ ]:
allowed_values = {
    "department": {"General Practice", "Physiotherapy", "Dermatology"},
    "appointment_type": {"Initial", "Follow-up"},
    "status": {"Completed", "Cancelled", "Missed"},
    "booking_channel": {"Phone", "Online", "Reception", "Unknown"},
}

for column, allowed in allowed_values.items():
    invalid_mask = clean_df[column].notna() & ~clean_df[column].isin(allowed)
    print(f"Invalid {column} values:", invalid_mask.sum())

## 18. Validate ranges and relationships

Our fictional administrative rules are:

- Waiting time cannot be negative.
- Scheduled duration must be between 5 and 180 minutes.
- Appointment date cannot be before booking date.
- Required identifiers cannot be missing and must match the expected pattern.

These checks are scenario-specific. A real project should confirm rules with the data owner.

In [ ]:
invalid_wait = clean_df["wait_minutes"].notna() & clean_df["wait_minutes"].lt(0)
invalid_duration = clean_df["scheduled_minutes"].notna() & ~clean_df["scheduled_minutes"].between(5, 180)
invalid_date_order = (
    clean_df["booking_date"].notna()
    & clean_df["appointment_date"].notna()
    & clean_df["appointment_date"].lt(clean_df["booking_date"])
)
invalid_id = ~clean_df["appointment_id"].str.fullmatch(r"APT-\d{4}", na=False)

print("Negative waiting times:", invalid_wait.sum())
print("Invalid scheduled durations:", invalid_duration.sum())
print("Appointment before booking:", invalid_date_order.sum())
print("Invalid appointment IDs:", invalid_id.sum())

In [ ]:
# Show records that fail at least one range or relationship rule.
rule_failure = invalid_wait | invalid_duration | invalid_date_order | invalid_id

clean_df.loc[
    rule_failure,
    [
        "appointment_id",
        "booking_date",
        "appointment_date",
        "wait_minutes",
        "scheduled_minutes",
    ],
]

### Preserve questionable values and add flags

A flag allows us to exclude invalid values from a particular calculation without deleting the whole record. It also keeps the evidence available for correction.

In [ ]:
clean_df["valid_wait"] = ~invalid_wait & clean_df["wait_minutes"].notna()
clean_df["valid_duration"] = ~invalid_duration & clean_df["scheduled_minutes"].notna()
clean_df["valid_date_order"] = ~invalid_date_order & clean_df["booking_date"].notna() & clean_df["appointment_date"].notna()
clean_df["valid_id"] = ~invalid_id

clean_df[[
    "appointment_id",
    "valid_id",
    "valid_wait",
    "valid_duration",
    "valid_date_order",
]].head()

## 19. Create row-level issue descriptions

A single `record_issues` field makes review easier. A record can contain more than one issue.

In [ ]:
def identify_record_issues(row):
    """Return a semicolon-separated description of data-quality issues."""
    issues = []

    if not row["valid_id"]:
        issues.append("invalid appointment ID")
    if pd.isna(row["booking_date"]):
        issues.append("missing or invalid booking date")
    if pd.isna(row["appointment_date"]):
        issues.append("missing or invalid appointment date")
    if pd.isna(row["wait_minutes"]):
        issues.append("missing or invalid wait minutes")
    elif not row["valid_wait"]:
        issues.append("negative wait minutes")
    if pd.isna(row["scheduled_minutes"]):
        issues.append("missing or invalid scheduled minutes")
    elif not row["valid_duration"]:
        issues.append("scheduled minutes outside allowed range")
    if (
        pd.notna(row["booking_date"])
        and pd.notna(row["appointment_date"])
        and not row["valid_date_order"]
    ):
        issues.append("appointment date before booking date")
    if pd.isna(row["reminder_sent"]):
        issues.append("missing reminder status")

    return "; ".join(issues) if issues else "No detected issues"


clean_df["record_issues"] = clean_df.apply(identify_record_issues, axis=1)
clean_df["record_valid"] = clean_df["record_issues"].eq("No detected issues")

clean_df[["appointment_id", "record_valid", "record_issues"]]

### Count affected records

A row-level flag supports transparent filtering. It does not declare that a record is useless; it states that the record failed at least one documented check.

In [ ]:
print(clean_df["record_valid"].value_counts())

record_quality_step(
    "Apply row-level validation",
    (~clean_df["record_valid"]).sum(),
    "Flagged records with missing, invalid, or inconsistent values; retained them for review.",
)

## 20. Use valid subsets for specific calculations

Suppose we want the average recorded waiting time. We should use values that are both present and non-negative. We do not need to discard an entire appointment because another field is missing.

In [ ]:
valid_waits = clean_df.loc[clean_df["valid_wait"], "wait_minutes"]

print("Valid waiting-time values:", len(valid_waits))
print(f"Mean valid wait: {valid_waits.mean():.1f} minutes")
print(f"Median valid wait: {valid_waits.median():.1f} minutes")

> Validation should match the question. A missing reminder status does not necessarily prevent a waiting-time calculation. A negative waiting time does.

## 21. Review the cleaned schema

Compare the final data types and categories with the original profile.

In [ ]:
clean_df.info()

In [ ]:
print("Departments:", sorted(clean_df["department"].dropna().unique()))
print("Statuses:", sorted(clean_df["status"].dropna().unique()))
print("Channels:", sorted(clean_df["booking_channel"].dropna().unique()))
print("Duplicate IDs:", clean_df["appointment_id"].duplicated().sum())

## 22. Review the data-quality log

The log turns invisible editing into an explicit analytical record.

In [ ]:
quality_log_df = pd.DataFrame(quality_log)
quality_log_df

## 23. Run final assertions

Assertions encode conditions that should be true after cleaning. If an assertion fails, the pipeline stops instead of quietly exporting an unexpected result.

In [ ]:
expected_columns = {
    "appointment_id",
    "booking_date",
    "appointment_date",
    "department",
    "appointment_type",
    "wait_minutes",
    "scheduled_minutes",
    "status",
    "reminder_sent",
    "booking_channel",
}

assert expected_columns.issubset(clean_df.columns)
assert clean_df["appointment_id"].is_unique
assert clean_df["appointment_id"].str.fullmatch(r"APT-\d{4}", na=False).all()
assert clean_df["department"].dropna().isin(allowed_values["department"]).all()
assert clean_df["status"].dropna().isin(allowed_values["status"]).all()
assert pd.api.types.is_datetime64_any_dtype(clean_df["booking_date"])
assert pd.api.types.is_datetime64_any_dtype(clean_df["appointment_date"])
assert not clean_df.duplicated().any()

print("All final structural assertions passed.")

> We intentionally do **not** assert that every row is fully valid. Invalid and missing values remain visible and are documented in the quality flags. The assertions confirm the properties the pipeline promised to enforce.

## 24. Export and verify

We export:

- The cleaned dataset, including validation flags.
- The data-quality log.

Then we read both files back to confirm that they exist and have the expected dimensions.

In [ ]:
clean_path = Path("fictional_appointments_clean.csv")
log_path = Path("fictional_appointments_quality_log.csv")

clean_df.to_csv(clean_path, index=False, date_format="%Y-%m-%d")
quality_log_df.to_csv(log_path, index=False)

print("Saved:", clean_path)
print("Saved:", log_path)

In [ ]:
verified_clean = pd.read_csv(clean_path)
verified_log = pd.read_csv(log_path)

print("Verified clean-data shape:", verified_clean.shape)
print("Verified log shape:", verified_log.shape)

display(verified_clean.head(3))
display(verified_log.head(3))

## 25. Code-along challenge: write a quality summary

Create a one-row summary with:

- Raw row count.
- Clean row count.
- Exact duplicates removed.
- Records with detected issues.
- Missing or invalid booking dates.
- Missing or invalid appointment dates.
- Missing or invalid scheduled durations.
- Negative waiting times.

Complete the cell below.

In [ ]:
# Your turn: replace the comments with calculations.
quality_summary = pd.DataFrame({
    # "raw_rows": [...],
    # "clean_rows": [...],
    # "exact_duplicates_removed": [...],
    # "records_with_issues": [...],
    # "missing_or_invalid_booking_dates": [...],
    # "missing_or_invalid_appointment_dates": [...],
    # "missing_or_invalid_scheduled_minutes": [...],
    # "negative_waiting_times": [...],
})

quality_summary

<details>
<summary><strong>Reveal one possible solution</strong></summary>

Run the next cell only after attempting the challenge.
</details>

In [ ]:
quality_summary_solution = pd.DataFrame({
    "raw_rows": [len(raw_df)],
    "clean_rows": [len(clean_df)],
    "exact_duplicates_removed": [len(raw_df) - len(clean_df)],
    "records_with_issues": [(~clean_df["record_valid"]).sum()],
    "missing_or_invalid_booking_dates": [clean_df["booking_date"].isna().sum()],
    "missing_or_invalid_appointment_dates": [clean_df["appointment_date"].isna().sum()],
    "missing_or_invalid_scheduled_minutes": [clean_df["scheduled_minutes"].isna().sum()],
    "negative_waiting_times": [(clean_df["wait_minutes"] < 0).fillna(False).sum()],
})

quality_summary_solution

## 26. Independent practice

Create a cleaned copy of the small service-request table below.

Your workflow should:

1. Preserve the raw table.
2. Convert placeholder strings to missing values.
3. Standardise `request_type` and `status`.
4. Convert `resolution_hours` to numeric data.
5. Convert `request_date` to datetime data.
6. Identify duplicate request IDs.
7. Flag negative resolution times.
8. Create a short quality log.

Do not silently drop invalid records.

In [ ]:
service_requests = pd.DataFrame({
    "request_id": ["REQ-01", "REQ-02", "REQ-03", "REQ-03", "REQ-04"],
    "request_date": ["2026-09-01", "2026-09-02", "bad date", "bad date", "2026-09-04"],
    "request_type": [" IT support ", "facilities", "IT Support", "IT Support", "FACILITIES"],
    "resolution_hours": ["2", "not recorded", "-1", "-1", "5"],
    "status": ["closed", "OPEN", "closed", "closed", "Closed "],
})

service_requests

In [ ]:
# Build your service-request cleaning workflow here.
# This cell is intentionally comment-only so the notebook runs top to bottom.

### Practice checklist

Before considering the task complete, confirm:

- The raw object was not overwritten.
- Category mappings are visible in the code.
- Conversion failures can be counted.
- Duplicate IDs are displayed before any removal.
- Negative values are flagged.
- The cleaned schema is inspected.
- Cleaning decisions are written in a log or Markdown explanation.

## 27. Test your knowledge!

Answer these questions without running new code first.

1. Why should raw data and cleaned data be stored separately?
2. What is the difference between a placeholder string and a genuine observed value?
3. What does `errors="coerce"` do in `pd.to_numeric()` or `pd.to_datetime()`?
4. Why must conversion failures be counted after coercion?
5. What is the difference between an exact duplicate and a repeated identifier?
6. Why is filling every missing numeric value with the mean risky?
7. Why did we retain a negative waiting time while also marking it invalid?
8. What does nullable Boolean data allow that ordinary `bool` may not express clearly?
9. Why can an appointment date be valid as a date but invalid relative to its booking date?
10. How can inconsistent categories affect an ML model?
11. What is the purpose of a data-quality log?
12. Why do final assertions improve a cleaning pipeline?

<details>
<summary><strong>Check your answers</strong></summary>

1. It preserves the original evidence and allows the workflow to be repeated or audited.
2. A placeholder represents absence according to a documented rule; an observed value carries recorded information.
3. It converts values that cannot be parsed into a missing value instead of raising an exception.
4. Otherwise invalid source values may disappear into missingness without being noticed.
5. An exact duplicate repeats every field; a repeated identifier may represent either duplication or conflicting records and requires investigation.
6. Mean filling can change distributions, reduce variability, and create values that were never observed.
7. Retaining it preserves evidence for correction; the flag prevents it from entering calculations that require non-negative waits.
8. It supports `True`, `False`, and a missing state.
9. Each date may be parseable on its own while their order violates the process rule.
10. Different spellings can become separate categories or features, fragmenting the evidence available to the model.
11. It records transformations, affected records, and reasons so the work can be reviewed and reproduced.
12. Assertions stop the workflow when promised structural conditions are not met.

</details>

## 28. Lesson recap

You have completed the **preparation and cleaning** stage of systematic EDA.

You can now:

- Preserve raw data and work on a copy.
- Profile messy text, missingness, duplicates, and inferred types.
- Standardise labels with string methods and explicit mappings.
- Convert numeric, date, and Boolean fields while exposing failures.
- Compare missing-data strategies before choosing one.
- Validate allowed categories, ranges, identifiers, and date relationships.
- Retain questionable records with transparent quality flags.
- Create an audit log and verify exported results.

In Module 2.3, we will filter, combine, group, and reshape tabular data to answer operational questions across rows and tables.

## Further reading

- [pandas: Working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- [pandas: Working with text data](https://pandas.pydata.org/docs/user_guide/text.html)
- [pandas: `to_numeric`](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html)
- [pandas: `to_datetime`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
- [pandas: `drop_duplicates`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html)

Use documentation to confirm a method's exact behaviour before applying it to important data.